## Section 9: Final Model Evaluation and Conclusion

### Model Evaluation

Having trained and tuned two classical models (Random Forest,
XGBoost) and one Neural Network (MLP), we now perform a final comparative analysis

This notebook will:
- Compare models using MAE, RMSE, and R² metrics.
- Analyze error distribution across different Challenge Rating tiers.
- Discuss the trade-offs between model complexity and predictive power.
- Provide a final recommendation for implementation.

In [6]:
import pandas as pd
import numpy as np
import joblib
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


X_test = pd.read_csv('../data/final_split/X_test_nn.csv')
y_test = pd.read_csv('../data/final_split/y_test_nn.csv').values.ravel()


rf_model = joblib.load('../models/classical_tuned/tuned_rf_model.pkl')
xgb_model = joblib.load('../models/classical_tuned/tuned_xgb_model.pkl')


class CRPredictor(nn.Module): # todo: maybe make a separate file for this?
    """
    Feedforward MLP for predicting D&D monster Challenge Rating.
    
    Args:
        input_dim:     Number of input features (27)
        hidden_layers: Tuple of neuron counts per hidden layer, e.g. (128, 64, 32)
        dropout:       Dropout rate applied after each hidden layer
    """
    
    def __init__(self, input_dim, hidden_layers=(128, 64, 32), dropout=0.3):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for neurons in hidden_layers:
            layers.extend([
                nn.Linear(prev_dim, neurons),
                nn.BatchNorm1d(neurons),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = neurons
        
        layers.append(nn.Linear(prev_dim, 1))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

model_nn = CRPredictor(
    input_dim=X_test.shape[1],
    hidden_layers=(256, 128, 64),  
    dropout=0.3  
)
model_nn.load_state_dict(torch.load('../models/nn_tuned/nn_tuned.pth'))
model_nn.eval()


rf_preds = rf_model.predict(X_test)
xgb_preds = xgb_model.predict(X_test)

with torch.no_grad():
    nn_preds = model_nn(torch.tensor(X_test.values, dtype=torch.float32)).numpy().ravel()

results_df = pd.DataFrame({
    'Actual': y_test,
    'Random Forest': rf_preds,
    'XGBoost': xgb_preds,
    'Neural Network': nn_preds
})

In [8]:
# Metric Comparison Summary
def get_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

comparison_table = pd.DataFrame({
    'Random Forest': get_metrics(y_test, rf_preds),
    'XGBoost': get_metrics(y_test, xgb_preds),
    'Neural Network': get_metrics(y_test, nn_preds)
}).T

print("Global Model Performance Table")
display(comparison_table.round(4))

Global Model Performance Table


,MAE,RMSE,R2
Random Forest,0.8819,1.3562,0.9511
XGBoost,0.8531,1.3279,0.9531
Neural Network,1.0787,1.6897,0.9241


### Conclusions and Findings


#### **Summary of Results:**
- 

#### **Best Model Details:**
- 

#### **Future Improvements:**
- 

#### **References:**
- 

#### **AI Disclosure:**
- 